# Ingest

## Imports

In [ ]:
import json
import sys
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np


## Paths

In [ ]:
_cwd = Path.cwd()
ROOT = _cwd if (_cwd / 'scripts').exists() else _cwd.parent
DATASET_DIR = ROOT / 'data' / 'numbers'
OUT_DIR = ROOT / 'data' / 'hand_poses_public'
OUT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_NAMES = ['zero','one','two','three','four','five','six','seven','eight','nine']


## Helpers

In [ ]:
def imread_unicode(path):
    try:
        data = np.fromfile(str(path), dtype=np.uint8)
        if data.size == 0:
            return None
        return cv2.imdecode(data, cv2.IMREAD_COLOR)
    except Exception:
        return None

def normalize_hand(landmarks):
    if landmarks.shape != (21, 3):
        return None
    if np.abs(landmarks).sum() < 1e-6:
        return None
    wrist = landmarks[0]
    mid_mcp = landmarks[9]
    palm = float(np.linalg.norm(mid_mcp - wrist))
    if palm < 0.02 or palm > 0.5:
        return None
    centered = landmarks - wrist
    return (centered / palm).reshape(-1).astype(np.float32)

def pick_primary_hand(result):
    if not result.multi_hand_landmarks:
        return None
    best = None
    best_score = -1.0
    for lm, hd in zip(result.multi_hand_landmarks, result.multi_handedness):
        pts = np.array([[p.x, p.y, p.z] for p in lm.landmark], dtype=np.float32)
        w = pts[:, 0].max() - pts[:, 0].min()
        h = pts[:, 1].max() - pts[:, 1].min()
        score = (w * w + h * h) ** 0.5 * float(hd.classification[0].score)
        if score > best_score:
            best_score = score
            best = (pts, hd.classification[0].label)
    return best


## Extract

In [ ]:
X, y = [], []
per_class = {n: 0 for n in LABEL_NAMES}
skipped = 0
mp_hands = mp.solutions.hands
with mp_hands.Hands(static_image_mode=True, max_num_hands=2, model_complexity=1, min_detection_confidence=0.4) as hands:
    for digit in range(10):
        cls_dir = DATASET_DIR / str(digit)
        if not cls_dir.exists():
            continue
        files = sorted(p for p in cls_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        print(f'digit {digit}: {len(files)} images')
        for fp in files:
            img = imread_unicode(fp)
            if img is None:
                skipped += 1
                continue
            res = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            pick = pick_primary_hand(res)
            if pick is None:
                skipped += 1
                continue
            pts, label = pick
            if label == 'Left':
                pts = pts.copy()
                pts[:, 0] = -pts[:, 0]
            feat = normalize_hand(pts)
            if feat is None:
                skipped += 1
                continue
            X.append(feat)
            y.append(digit)
            per_class[LABEL_NAMES[digit]] += 1
            pts_m = pts.copy()
            pts_m[:, 0] = -pts_m[:, 0]
            feat_m = normalize_hand(pts_m)
            if feat_m is not None:
                X.append(feat_m)
                y.append(digit)
                per_class[LABEL_NAMES[digit]] += 1


## Save

In [ ]:
X = np.stack(X, 0).astype(np.float32)
y = np.array(y, dtype=np.int64)
np.save(OUT_DIR / 'X.npy', X)
np.save(OUT_DIR / 'y.npy', y)
(OUT_DIR / 'labels.json').write_text(json.dumps({str(i): n for i, n in enumerate(LABEL_NAMES)}, indent=2))
print(f'Saved {X.shape[0]} samples; skipped {skipped} images')
for n in LABEL_NAMES:
    print(f'  {n:>6}: {per_class[n]:>4}')
